# 🧪 Exploración Inicial: Schumann-Proteostasis Coupling

Este notebook interactivo permite explorar visualmente el comportamiento del modelo bajo diferentes parámetros físicos.

**Objetivos**:
- Visualizar el paisaje energético de doble pozo
- Simular trayectorias estocásticas
- Explorar el efecto del acoplamiento (η) y ruido (σ)
- Calcular métricas de estabilidad (MFPT)

In [ ]:
# ============================================================================
# IMPORTS Y CONFIGURACIÓN
# ============================================================================
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
from model import SchumannProteostasisModel

# Configuración de estilo
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print('✓ Módulos cargados correctamente')

## 🔧 Parámetros del Modelo

Ajusta estos valores para explorar diferentes regímenes físicos:

In [ ]:
# Parámetros base (puedes modificarlos)
params = {
    'eta': 0.5,           # Acoplamiento [0, 1]: 0=aislado, 1=poroso
    'sigma': 0.3,         # Intensidad de ruido biológico
    'A_schumann': 0.5,    # Amplitud del campo EM
    'f_schumann': 7.83,   # Frecuencia de Schumann (Hz)
    'T': 10.0,            # Duración de simulación (segundos)
    'dt': 0.001,          # Paso de tiempo
    'n_trials': 50,       # Número de trayectorias
    'seed': 42            # Semilla para reproducibilidad
}

print(f"Configuración: η={params['eta']}, σ={params['sigma']}, f={params['f_schumann']} Hz")

## 🧮 Inicialización del Modelo

In [ ]:
# Crear instancia del modelo
model = SchumannProteostasisModel(params)

# Encontrar atractores
well_healthy, well_pathological = model.find_well_positions()

print(f"🎯 Atractores identificados:")
print(f"   • Pozo saludable: x = {well_healthy:.3f}")
print(f"   • Pozo patológico: x = {well_pathological:.3f}")

## 📈 Visualización 1: Paisaje Energético

In [ ]:
x = np.linspace(-2, 2, 500)
V = model.potential(x)

plt.figure(figsize=(9, 5))
plt.plot(x, V, 'b-', linewidth=2.5, label='V(x)')
plt.axhline(y=0, color='k', linestyle='--', alpha=0.3)
plt.axvline(x=0, color='k', linestyle=':', alpha=0.3)
plt.plot(well_healthy, model.potential(well_healthy), 'go', markersize=10, label='Saludable')
plt.plot(well_pathological, model.potential(well_pathological), 'ro', markersize=10, label='Patológico')
plt.xlabel('Estado proteostático x', fontsize=12)
plt.ylabel('Energía potencial V(x)', fontsize=12)
plt.title('Paisaje Energético de Proteostasis', fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 🌊 Visualización 2: Campo de Schumann

In [ ]:
t_plot = np.linspace(0, 2, 1000)
field = model.schumann_field(t_plot)

plt.figure(figsize=(9, 4))
plt.plot(t_plot, field, 'b-', linewidth=2)
plt.xlabel('Tiempo (s)', fontsize=11)
plt.ylabel('Amplitud F_SR(t)', fontsize=11)
plt.title(f'Resonancia de Schumann: {model.f_SR} Hz', fontsize=13, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 🔄 Visualización 3: Trayectorias Estocásticas

In [ ]:
# Simular desde el pozo saludable
trajectories = model.simulate_euler_maruyama(x0=well_healthy, n_trials=params['n_trials'])

# Visualizar subset de trayectorias
plt.figure(figsize=(11, 6))
for i in range(min(20, trajectories.shape[0])):
    plt.plot(model.t, trajectories[i, :], alpha=0.5, linewidth=0.8)

plt.axhline(y=well_healthy, color='g', linestyle='--', alpha=0.6, label='Pozo saludable')
plt.axhline(y=well_pathological, color='r', linestyle='--', alpha=0.6, label='Pozo patológico')
plt.axhline(y=0, color='k', linestyle=':', alpha=0.3)
plt.xlabel('Tiempo (s)', fontsize=12)
plt.ylabel('Estado x(t)', fontsize=12)
plt.title(f'Trayectorias de Proteostasis\nη={params["eta"]}, σ={params["sigma"]}', fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 📊 Métricas de Estabilidad

In [ ]:
# Calcular MFPT (Mean First Passage Time)
mfpt, success_rate = model.calculate_mfpt(trajectories, threshold=0.0)

print(f"📈 Métricas calculadas:")
print(f"   • MFPT: {mfpt:.3f} segundos")
print(f"   • Tasa de transición: {success_rate:.1%}")
print(f"")
if mfpt > 5.0:
    print("✅ Sistema ALTAMENTE ESTABLE bajo estas condiciones")
elif mfpt > 2.0:
    print("⚠️ Sistema MODERADAMENTE ESTABLE")
else:
    print("❌ Sistema PROPENSO a transiciones patológicas")

## 🎛️ Experimento Rápido: Variar η (Acoplamiento)

In [ ]:
# Barrido rápido sobre el parámetro eta
eta_values = [0.1, 0.3, 0.5, 0.7, 0.9]
results = []

print("Explorando efecto del acoplamiento η sobre la estabilidad...\n")

for eta in eta_values:
    params_test = params.copy()
    params_test['eta'] = eta
    params_test['T'] = 5.0  # Simulación más corta para el barrido
    
    model_test = SchumannProteostasisModel(params_test)
    well1, _ = model_test.find_well_positions()
    traj = model_test.simulate_euler_maruyama(x0=well1, n_trials=30)
    mfpt_test, success_test = model_test.calculate_mfpt(traj, threshold=0.0)
    
    results.append({'eta': eta, 'mfpt': mfpt_test, 'success': success_test})
    print(f"η={eta:.1f} → MFPT={mfpt_test:.2f}s, transiciones={success_test:.0%}")

# Visualizar resultados
etas = [r['eta'] for r in results]
mfpts = [r['mfpt'] for r in results]

plt.figure(figsize=(8, 5))
plt.plot(etas, mfpts, 'bo-', linewidth=2, markersize=8)
plt.xlabel('Acoplamiento η', fontsize=12)
plt.ylabel('MFPT (s)', fontsize=12)
plt.title('Estabilidad vs Acoplamiento Ambiental', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 💡 Próximos Pasos

1. **Variar σ (ruido)**: Explorar resonancia estocástica
2. **Barrido 2D**: Heatmap de estabilidad en el espacio (η, σ)
3. **Exportar datos**: Guardar resultados para análisis estadístico
4. **Comparar frecuencias**: Probar f ≠ 7.83 Hz (desintonía)

```python
# Ejemplo: guardar resultados
import json
with open('../results/processed/experiment_001.json', 'w') as f:
    json.dump({'params': params, 'mfpt': mfpt, 'success_rate': success_rate}, f, indent=2)
```